# Baseline embedding metrics
DINOv2-large + CLIP-large image similarity (generated vs each reference, averaged) on the baseline images. Deterministic, no API.

In [ ]:
# mount Drive, paths; restore reference crops from tar if absent
from google.colab import drive
import os, subprocess
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
GEN   = f"{DRIVE}/gen/baseline"
IMGS, EXPECTED = "/content/imgs", 37063
TAR   = f"{DRIVE}/imgs_subset.tar"

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0
if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}', shell=True, check=True)
print("reference images:", n_files(IMGS))
if not os.path.exists("/content/test.jsonl"):
    subprocess.run(["cp", f"{DRIVE}/test.jsonl", "/content/test.jsonl"])
print("baseline images:", len([f for f in os.listdir(GEN) if f.endswith(".png")]) if os.path.isdir(GEN) else "MISSING")

Mounted at /content/drive
reference images: 37063
baseline images: 488


In [ ]:
# load DINOv2-large + CLIP-large; image embedding = normalized CLS / projected feature
import torch, torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel, CLIPImageProcessor, CLIPVisionModelWithProjection

device = "cuda" if torch.cuda.is_available() else "cpu"
dino_proc = AutoImageProcessor.from_pretrained("facebook/dinov2-large")
dino      = AutoModel.from_pretrained("facebook/dinov2-large").to(device).eval()
clip_proc = CLIPImageProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip      = CLIPVisionModelWithProjection.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()

@torch.no_grad()
def dino_emb(imgs):                       # list[PIL] -> (N,D) L2-normalized CLS tokens
    inp = dino_proc(images=imgs, return_tensors="pt").to(device)
    return F.normalize(dino(**inp).pooler_output.float(), dim=-1)

@torch.no_grad()
def clip_emb(imgs):                       # list[PIL] -> (N,D) L2-normalized projected image features
    inp = clip_proc(images=imgs, return_tensors="pt").to(device)
    return F.normalize(clip(**inp).image_embeds.float(), dim=-1)

def sims(gen, refs):                      # cosine gen-vs-each-ref, averaged
    d = dino_emb([gen] + refs); c = clip_emb([gen] + refs)
    return (d[0:1] @ d[1:].T).mean().item(), (c[0:1] @ c[1:].T).mean().item()

print("device:", device, "| dino dim:", dino.config.hidden_size, "| clip dim:", clip.visual_projection.out_features)

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/439 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.52k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{

device: cuda | dino dim: 1024 | clip dim: 768


In [ ]:
# per-row DINO + CLIP-I (gen vs each reference, averaged); resumable
import json, os
from PIL import Image
rows = [json.loads(l) for l in open("/content/test.jsonl", encoding="utf-8")]
EMB = f"{DRIVE}/scores/baseline_embed.jsonl"
os.makedirs(os.path.dirname(EMB), exist_ok=True)

done = set()
if os.path.exists(EMB):
    for l in open(EMB, encoding="utf-8"):
        try:
            d = json.loads(l)
            if "error" not in d: done.add(d["i"])
        except: pass

todo = [i for i in range(len(rows)) if i not in done]
print(f"to embed: {len(todo)} (done: {len(done)})")
with open(EMB, "a", encoding="utf-8") as f:
    for n, i in enumerate(todo, 1):
        r = rows[i]; gp = f"{GEN}/test_{i:05d}.png"
        if not os.path.exists(gp):
            f.write(json.dumps({"i": i, "error": "missing_generated"}) + "\n"); continue
        gen  = Image.open(gp).convert("RGB")
        refs = [Image.open(os.path.join(IMGS, p)).convert("RGB") for p in r["input_images"]]
        ds, cs = sims(gen, refs)
        ups = ["_up" in os.path.basename(p) for p in r["input_images"]]
        f.write(json.dumps({"i": i, "uid": r["output_image"] + "|" + "|".join(sorted(r["input_images"])),
                            "n_refs": len(refs), "ref_kind": "all_up" if all(ups) else ("no_up" if not any(ups) else "mixed"),
                            "dino": round(ds, 4), "clip": round(cs, 4)}) + "\n")
        f.flush()
        if n % 50 == 0: print(f"  {n}/{len(todo)}")
print("done ->", EMB)

to embed: 0 (done: 488)
done -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/scores/baseline_embed.jsonl


In [ ]:
# aggregate + correlation with judge SC
import json, math
from collections import defaultdict
EMB = f"{DRIVE}/scores/baseline_embed.jsonl"
BP  = f"{DRIVE}/scores/test__baseline.jsonl"

recs = [json.loads(l) for l in open(EMB, encoding="utf-8")]
recs = [d for d in recs if "error" not in d]

def stat(rs, key): v = [r[key] for r in rs]; return sum(v) / len(v)
def table(key):
    g = defaultdict(list)
    for r in recs: g[r[key]].append(r)
    for k in sorted(g, key=str):
        print("  %-8s n=%3d  DINO=%.3f  CLIP=%.3f" % (str(k), len(g[k]), stat(g[k], "dino"), stat(g[k], "clip")))

print("rows:", len(recs))
print("POOLED   DINO=%.3f  CLIP=%.3f\n" % (stat(recs, "dino"), stat(recs, "clip")))
print("by reference count:"); table("n_refs")
print("\nby provenance:");    table("ref_kind")

# correlation with judge SC (validation that embeddings track subject consistency)
if os.path.exists(BP):
    sc = {}
    for l in open(BP, encoding="utf-8"):
        d = json.loads(l)
        if "error" not in d: sc[d["i"]] = d["SC"]
    def pearson(xs, ys):
        n = len(xs); mx = sum(xs)/n; my = sum(ys)/n
        cov = sum((x-mx)*(y-my) for x, y in zip(xs, ys))
        sx = math.sqrt(sum((x-mx)**2 for x in xs)); sy = math.sqrt(sum((y-my)**2 for y in ys))
        return cov/(sx*sy) if sx and sy else float("nan")
    common = [r for r in recs if r["i"] in sc]
    if common:
        scv = [sc[r["i"]] for r in common]
        print("\ncorrelation with judge SC (n=%d):  DINO r=%.3f  CLIP r=%.3f"
              % (len(common), pearson([r["dino"] for r in common], scv), pearson([r["clip"] for r in common], scv)))

rows: 488
POOLED   DINO=0.435  CLIP=0.760

by reference count:
  1        n=243  DINO=0.436  CLIP=0.761
  2        n=194  DINO=0.433  CLIP=0.760
  3        n= 51  DINO=0.438  CLIP=0.760

by provenance:
  all_up   n=198  DINO=0.459  CLIP=0.763
  mixed    n=133  DINO=0.413  CLIP=0.763
  no_up    n=157  DINO=0.423  CLIP=0.755

correlation with judge SC (n=488):  DINO r=0.039  CLIP r=0.108


In [ ]:
# diagnose: does the embedding carry usable subject signal, or is it range restriction?
import json, statistics as st
from scipy.stats import pearsonr, spearmanr

DRIVE = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
emb  = {d["i"]: d for d in (json.loads(l) for l in open(f"{DRIVE}/scores/baseline_embed.jsonl", encoding="utf-8")) if "error" not in d}
base = {d["i"]: d for d in (json.loads(l) for l in open(f"{DRIVE}/scores/test__baseline.jsonl", encoding="utf-8")) if "error" not in d}
I = [i for i in emb if i in base]

def col(key, src, idx): return [src[i][key] for i in idx]
def corr(xs, ys): return float(pearsonr(xs, ys)[0]), float(spearmanr(xs, ys).statistic)

print("per-row spread:  DINO std=%.3f  CLIP std=%.3f" % (st.pstdev(col("dino", emb, I)), st.pstdev(col("clip", emb, I))))

def report(idx, label):
    sc = col("SC", base, idx)
    dp, ds = corr(col("dino", emb, idx), sc)
    cp, cs = corr(col("clip", emb, idx), sc)
    print("%-22s n=%3d | DINO pearson=%.3f spearman=%.3f | CLIP pearson=%.3f spearman=%.3f"
          % (label, len(idx), dp, ds, cp, cs))

report(I, "all rows")
report([i for i in I if base[i]["SC"] <= 7], "non-ceiling (SC<=7)")
report([i for i in I if emb[i]["n_refs"] == 1], "1-ref only")

per-row spread:  DINO std=0.176  CLIP std=0.090
all rows               n=488 | DINO pearson=0.039 spearman=0.015 | CLIP pearson=0.108 spearman=0.105
non-ceiling (SC<=7)    n= 86 | DINO pearson=0.246 spearman=0.225 | CLIP pearson=0.158 spearman=0.144
1-ref only             n=243 | DINO pearson=0.080 spearman=0.075 | CLIP pearson=0.159 spearman=0.178
